In [0]:
# ─── 00_imports — Librairies et configuration globale ───────────────────────
# Appelé par tous les notebooks via : %run ./00_imports
# ────────────────────────────────────────────────────────────────────────────

import requests
import json
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StringType, DoubleType,
    DateType, IntegerType
)
from pyspark.sql.window import Window
from datetime import datetime

print("Librairies chargées")

In [0]:

CONFIG_PATH = "/Workspace/Users/sabineanoko@gmail.com/pipeline-qualite-eau/config/pipeline_config.json"

with open(CONFIG_PATH, "r") as f:
    CONFIG = json.load(f)

# Raccourcis pratiques
DATASET_ID   = CONFIG["dataset"]["id"]
DATASET_SLUG = CONFIG["dataset"]["slug"]
FICHIERS = CONFIG["fichiers"]
API_BASE_URL = CONFIG["dataset"]["api_base_url"]
ANNEES       = CONFIG["dataset"]["annees"]
CATALOG      = CONFIG["catalog"]["name"]
TABLES       = CONFIG["tables"]

print(f"Config chargée depuis : {CONFIG_PATH}")
print(f"Années ciblées : {ANNEES}")
print(f"Catalog : {CATALOG}")
print(f"Tables configurées : {list(TABLES.keys())}")

In [0]:
# ─── Widgets — paramètres dynamiques ─────────────────────────────────────────
# Les widgets apparaissent en haut du notebook comme des menus déroulants.

dbutils.widgets.removeAll()  # Nettoie les widgets existants

dbutils.widgets.dropdown(
    "annee", 
    "2024", 
    [str(a) for a in ANNEES], 
    "Année cible"
)

dbutils.widgets.dropdown(
    "env",
    "dev",
    ["dev", "prod"],
    "Environnement"
)

# Récupérer les valeurs
ANNEE_CIBLE = int(dbutils.widgets.get("annee"))
ENV         = dbutils.widgets.get("env")

print(f"Widgets configurés")
print(f"Année sélectionnée : {ANNEE_CIBLE}")
print(f"Environnement : {ENV}")

In [0]:

# Créer les schémas s'ils n'existent pas
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")